# Deliverable 2
2026-06-21

Begin test assessments and complete Group 1 "required" tests.

| Stage | Stage specifications | Time required | Start-End |
| ----- | -------------------- | ------------- | --------- |
| 1 | Learn about the community and note shortcomings or additional toolboxes, such that the wheel need not be reinvented. | 3 weeks | May 1 - May 24 |
| **2** | **Begin test assessments and complete Group 1 “required” tests.** | **4 weeks** | **May 25 - June 21** |
| 3 | Complete Group 2 “Strongly Recommended” tests. | 2 weeks | June 22 - July 5 |
| 4 | Complete Group 3 “Suggested” tests. | 3 weeks | July 6 - July 26 |
| 5 | Explore other tests and finalize documentation.| 3 weeks | July 27 - August 16 |

Therefore, the objectives for this deliverable are to assess each of the following tests in both the QARTOD data manual and in the toolbox itself.

1. Timing/Gap Test
2. Syntax Test
3. Location Test
4. Gross Range Test
    * Including similar Climatological Test (strongly recommended)
    * "Seasonal expectations" variation on the gross range test.
    * Allen et al. 2012 - Variability Generalixed Digital Environmental Model
    * Boyer et al. 2013 - WOD
5. Pressure Test
    * Including similar Density Inversion Test (suggested)
    * Pressure is looking for a monotonically ordered record.
    * This is distinguished by checking potential density $\sigma _{\theta}$ increases with increasing depth. Easily done with `gsw`.
    * Could bundle this with test 8 - Rate of Change.

In [1]:
#   Imports of required packages/modules/libraries for use in this deliverable's tweaks.
#   For handling more generic types (see https://docs.python.org/3/library/stdtypes.html)
from typing import TYPE_CHECKING
if TYPE_CHECKING:
    from collections.abc import Sequence    #   list, tuple, range are the three basic sequence types.
    from numbers import Real                #   Real numbers, including `int` and `float` but not complex or imaginary numbers.

import numpy as np
import xarray as xr

In [2]:
#   Import a testing dataset
#   OG1 is OK?
fpath = "/home/aaron-mau/Data/OG1/delayed_SEA056_M102.nc"
ds = xr.load_dataset(fpath)

In [3]:
class QartodFlags:
    """Primary flags for QARTOD."""

    GOOD = 1
    UNKNOWN = 2
    SUSPECT = 3
    FAIL = 4
    MISSING = 9
FLAGS = QartodFlags  # Default name for all check modules
NOTEVAL_VALUE = QartodFlags.UNKNOWN

## Timing/Gap Test
* How is it described in the manual
* How is it represented in the code
* Does anything need to be done in either?

### Manual description of timing/gap test
It's a **required** test, so it's basically one of the most important checks that can be done on the data. Which makes sense - time is one of the dimensions of the data. If you don't know when the data was taken, it loses much of its value.

The manual describes it as "Check for arrival of data"
> Test determines that the most recent profile has been received within the expected time window
> (TIM_INC) and has the correct time stamp (TIM_STMP).
> 
> **Note:** For those gliders that do not update at regular intervals, a large value for TIM_STMP can be
> assigned. The gap check is not a solution for all timing errors. Data could be measured or received
> earlier than expected. This test does not address all clock drift/jump issues.

This is looking at expected time windows `(TIME_INC)` and has an appropriate time stamp `(TIM_STMP)`. In the example, `TIM_INC` is set to 6 hours.

Only 2 flags to assign: Either it passes the test (flag = 1) or it fails by not falling within the specified reporting range (flag = 4).

> `if NOW - TIM_STMP > TIM_INC, flag = 4`

In the manual, it isn't super clear what NOW is supposed to represent. It could be representing real-time applications, but in reality, it's probably just looking at the sequential data points. `NOW - TIM_STMP` is probably meant to be representing sequential data points, rather than whatever `TIM_STMP` value is supposed to be.

In this confusion, I had a discussion with one of my mentors. While it is doing two things, this doesn't apply to instances *not* in a real-time application. The fact that it suggests it is doing two tests in one could create compounding problems:
* The data has a good timestamp and is received in the designated time window (two successes, passes)
* The data has a good timestamp but isn't received in the designated time window (one success, fails)
* The data has a **bad** timestamp and isn't received in the designated time window (one success, fails)
* The data has a **bad** timestamp but **is** received within the time window (no successes, fails)

### Code for timing/gap test
I did a quick search for the manual terms, and there's already a disconnect. There is no term for `TIM_STMP` or `TIM_INC` in the entire `ioos_qc` package. I think this is worth bringing up, as users will probably investigate in the same way that I am now. Looking for the terminology that is described in the manual within the code to see how the parameters match up.

Furthermore, there is no defined function or class within `qartod.py` that describes something representing a timing or gap test.
* `utils.py` has a "check_timestamps" function. However, the docstring explicitly says "This is not a QARTOD test, but rather a utility test to make sure **times are in the proper order** and optionally **do not have large gaps prior to processing the data**. It takes in a `max_time_interval` value and returns a boolean. It does not return flags - rather a single boolean of pass/fail.

**Question for Filipe**: We want to reuse code as much as possible, as defined in the GSoC project proposal. Does it make sense to "graduate" this function, or are we sacrificing utility? This `utils.check_timestamps` function is called in `test_utils.py`, so I don't know how it is actually used (if at all).
* It also *adds* a function of confirming that the times are in the proper order. This may not be desireable or in the scope of this description in the manual. **Question for Filipe**: Is that itself worthy of an additional test?

**Question for Filipe**: This is running on time, which is often a coordinate. Do you envision needing to do anything differently here? 

**Question for Filipe**: Is there a timestamp format that we should be expecting? Should it be homogenous in the notebook?

### Actions for timing/gap test
* (For external workgroup) I think that the manual's description of the test could be improved. "Check for arrival of data" $\rightarrow$ "Check data reporting period".
    * Filipe agrees. The QARTOD working group should go over what this test is intended for in the manual and consider a revision to prevent future confusion.
* Build a new series of tests that fits the description in the manual (without having compounding problems), and an additional test that checks for gaps *within the data itself*. This means:
    1. `impossible_date_test` (similar to GSTPP test 1.2 - timestamps shouldn't be prehistoric) (this should only be flags 1 and 4)
    2. `data_reception_test` (timestamps should be recent for NRT applications) (as per the manual, this is only flags 1 and 4)
    3. `time_gap_test` (designate a window of a certain size and flag points following said gap)

#### impossible_date_test()
Since this is my first change that I'll be proposing to the QARTOD QC, we'll want to preserve the existing codebase and make sure that everything is consistent. This means the docstrings at the beginning of the function should be similar to other existing tests, typing should be consistent, and the flag exports should be the same general item.

There are a few notes to make prior to starting this test, however.

##### Numpy masked arrays
Most of the other tests use numpy **masked** arrays, of which I wasn't particularly familiar with.

[Masked arrays](https://numpy.org/doc/stable/reference/maskedarray.generic.html) seem to be handy because they consist of the data and a mask. Kind of like a way to track a condition on each point in a data array.

In the documentation, the following example is given:
```python
import numpy as np
import numpy.ma as ma
y = ma.array([1, 2, 3], mask = [0, 1, 0])
```

In this case, the second point (index 1) is 1, meaning that this element is invalid. It's a great way to keep track of tests that are run on data (pass/fail, yes/no).

These arrays are commonly referenced using `y.data` or `y.mask` for the values and mask, respectively. Cool! I haven't worked very much with this before and it's much simpler than assigning multiple ndarrays to handle the same thing.

##### Regarding datetime64
[Numpy's datetime64](https://numpy.org/doc/stable/reference/arrays.datetime.html) is handy for a lot of things, but keep in mind that it is usually formatted as a timestamp *since Jan 1st, 1970*. The year, month, day, and minute can all be pulled out with some simple string formatting (like `"datetime64[Y]"), but you'll need to convert these items relative to 1970. *Note that you should convert it to an integer before using it.*
* datetime64[Y] $\rightarrow$ add '1970' to it.
* datetime64[M] $\rightarrow$ mod 12 and add 1 for the calendar month.
* datetime64[h] $\rightarrow$ mod 24 gives you the hour of the day.
* datetime64[m] $\rightarrow$ mod 60 gives you the minute of the hour.

For the dates in the month, you've got cases like leap years which we'll want to account for which adds a 29th day to February. We can get how long each month is using each calendar month, converting that to the day, and doing the same for the next month's start. Then the difference between them should be how long the month is, and which valid days there are.

`first day of March - first day of February = length of February`

Furthermore, we want the day of the month to be valid. We could approach this from a complicated series of if statements based on Julian date and the current year, calculating out each month independently, but we can also convert from months to days using Numpy's `astype` arguments.

In [ ]:
def impossible_date_test(
        tinp: Sequence[Real],
        fail_span: tuple[Real, Real] | None = None,
) -> np.ma.core.MaskedArray:
    """
    This test confirms that the date and time for the data are reasonable.
    
    Given an array of time data, this test breaks the data down into a series of sub-tests.
    These tests are similar to those outlined in test 1.2 of the GTSPP RTQC Manual (IOC, 2010).
    * The year is either present or in the past.
    * The month is within 1-12.
    * The day is possible for the given month, depending on year.
    * The hour of the day is between 0-23.
    * The minutes are between 0-50.
    * (Optional) The time data is within a user-defined tolerance.
    Data deemed to have failed any or all of these tests are flagged as FAIL. Any missing and masked data is flagged as UNKNOWN.

    Parameters
    ----------
    tinp
        Time input data as a numeric numpy array or list of real numbers.
    fail_span
        2-tuple range which to flag outside data as FAIL. [optional]

    Returns
    -------
    flag_arr
        A masked array of flag values equal in size to that of the input `tinp`.
    """
    #   Init 
    original_shape = tinp.shape
    tinp = np.ma.asarray(tinp, dtype="datetime64[ns]").flatten()
    flag_arr = np.ma.ones(tinp.size, dtype="uint8") #   Init to flag 1 "good"
    
    tinp.mask = np.isnat(tinp.data)
    flag_arr[tinp.mask] = QartodFlags.MISSING   #   Init missing timestamps to the missing flag
    valid = ~tinp.mask  #   Define where the data point are not missing, such that we can index them and keep those flags.

    #   Year test - where year is in the future
    yr = tinp.data.astype("datetime64[Y]").astype(int) + 1970
    current_yr = np.datetime64("now", "Y").astype(int) + 1970
    flag_arr[valid & (yr > current_yr)] = QartodFlags.FAIL

    # #   Month test - the month must be 1-12
    # mn = tinp.data.astype("datetime64[M]").astype(int) % 12 + 1
    # flag_arr[valid & ((mn < 1) | (mn > 12))] = QartodFlags.FAIL
    
    # #   Day test - day must be possible for the given month
    # #   Define the start of the first and next month for all points in the array
    # mn_1 = tinp.data.astype("datetime64[M]").astype("datetime64[D]")
    # mn_2 = (tinp.data.astype("datetime64[M]") + 1).astype("datetime64[D]")
    # days = (mn_2 - mn_1).astype(int)
    # #   Day of the month
    # dy   = (tinp.data.astype("datetime64[D]") - mn_1).astype(int) + 1
    # flag_arr[valid & ((dy < 1) | (dy > days))] = QartodFlags.FAIL

    # #   Hour test - must be 0-23
    # hr = tinp.data.astype("datetime64[h]").astype(int) % 24
    # flag_arr[valid & ((hr < 0) | (hr > 23))] = QartodFlags.FAIL

    # #   Minute test - must be 0-59
    # mi = tinp.data.astype("datetime64[m]").astype(int) % 60
    # flag_arr[valid & ((mi < 0) | (mi > 59))] = QartodFlags.FAIL

    #   Span test
    if fail_span is not None:
        #   Convert if not already
        low, high = np.datetime64(fail_span[0]), np.datetime64(fail_span[1])
        flag_arr[valid & ((tinp.data < low) | (tinp.data > high))] = QartodFlags.FAIL
    
    return flag_arr.reshape(original_shape)

Discovery while testing: Numpy datetime64 doesn't actually allow months, days, hours, or minutes to be outside of tolerated ranges.

Inserting a fake datetime throws a ValueError:
```
np.datetime64("2025-12-32")
ValueError: Day out of range in datetime string "2025-12-32"

np.datetime64("2026-01-12T17:61:07.000000000")
ValueError: Minutes out of range in datetime string "2026-01-12T17:61:07.000000000"
```
As such, we can comment out many of the checks we defined. It even handles leap days.

```
np.datetime64("2024-02-29") #   This was a leap day

np.datetime64("2025-02-29") #   Most certainly not a leap day
ValueError: Day out of range in datetime string "2025-02-29"
```
The minutia described in the GTSPP is handled by datetime64 - if there are impossible values with the date and time then it will error out before getting tested and that tends to be easy to find.

Therefore, the most important aspect of this is probably the span test and checking for the future year. This can be further simplified by just looking for dates in the future, so we won't need to break out the year at all.

The final function to be added should then be

In [38]:
def impossible_date_test(
        tinp: Sequence[Real],
        fail_span: tuple[Real, Real] | None = None,
) -> np.ma.core.MaskedArray:
    """
    This test confirms that the date and time for the data are reasonable.
    
    Given an array of time data, this test breaks the data down into a series of sub-tests.
    These tests are similar to those outlined in test 1.2 of the GTSPP RTQC Manual (IOC, 2010).
    * The year is either present or in the past.
    * (Optional) The time data is within a user-defined tolerance.
    Data deemed to have failed any or all of these tests are flagged as FAIL. Any missing and masked data is flagged as UNKNOWN.

    Parameters
    ----------
    tinp
        Time input data as a numeric numpy array or list of real numbers.
    fail_span
        2-tuple range which to flag outside data as FAIL. [optional]

    Returns
    -------
    flag_arr
        A masked array of flag values equal in size to that of the input `tinp`.
    """
    #   Init 
    original_shape = tinp.shape
    tinp = np.ma.asarray(tinp, dtype="datetime64[ns]").flatten()
    flag_arr = np.ma.ones(tinp.size, dtype="uint8") #   Init to flag 1 "good"
    
    tinp.mask = np.isnat(tinp.data)
    flag_arr[tinp.mask] = QartodFlags.MISSING   #   Init missing timestamps to the missing flag
    valid = ~tinp.mask  #   Define where the data point are not missing, such that we can index them and keep those flags.

    #   Check for time travelers
    now = np.datetime64("now")
    flag_arr[valid & (tinp.data > now)] = QartodFlags.FAIL

    #   Span test
    if fail_span is not None:
        #   Convert if not already
        low, high = np.datetime64(fail_span[0]), np.datetime64(fail_span[1])
        flag_arr[valid & ((tinp.data < low) | (tinp.data > high))] = QartodFlags.FAIL
    
    return flag_arr.reshape(original_shape)

Let's confirm that the checker works. Primarily, this is confirming that there are no NaT values, future dates, or anything outside of our span.

In [ ]:
flags = impossible_date_test(ds.TIME.to_numpy(), fail_span = ("2020-01-01T00:00:00.0000", "2027-01-01T00:00:00.0000"))
print(any(flags == QartodFlags.FAIL))
print(any(flags == QartodFlags.MISSING))

False
False


Let's insert a bad data point in the future and confirm that we get one bad point out.

In [19]:
bad_data = ds.TIME.to_numpy().copy()
bad_data[20000] = np.datetime64("2028-01-12T23:05:16.000000000")

In [ ]:
flags = impossible_date_test(bad_data)
print(any(flags == QartodFlags.FAIL))
print(np.where(flags == QartodFlags.FAIL)[0][0])    #   Return the index - np.where nests things in a tuple of arrays

True
20000


And we'll modify the data further by inserting a chunk of dates that are outside of our `fail_span`. We'll insert a lot, so let's just return the first and last 5 indexes where the flags are bad. We should still see the bad time from before at index 20000.

In [40]:
bad_data[1500:2200] = np.datetime64("2013-01-12T23:05:16.000000000")
flags = impossible_date_test(bad_data, fail_span=("2020-01-01T00:00:00.0000","2027-01-01T00:00:00.0000"))
idx = np.where(flags == QartodFlags.FAIL)[0]
print(idx[:5],idx[-5:])

[1500 1501 1502 1503 1504] [ 2196  2197  2198  2199 20000]


#### data_reception_test()
This is a pretty straightforward test that should be more useful to datacenters that are expecting data within a certain timeframe. In many ways, it seems like a subset of the previous test. In the manual, the time window is described as `TIM_INC`, where the example sets it to 6 hours. In this case, our `fail_span` can be a single value, in hours, which if the data is older than by that amount, will cause it to be flagged.

It might be helpful to define a `starting timestamp` value of some sort (I'll call it `from_time`), such that we can check if data were received within X hours of timestamp Y. This isn't explicitly stated in the manual, so let's have it be optional and default to `np.datetime64("now")`.

In [46]:
def data_reception_test(
        tinp: Sequence[Real],
        fail_span: Real = 6,
        from_time: Real | None = None,
) -> np.ma.core.MaskedArray:
    """
    This test checks for data timestamps to be within a certain amount of time of present.
    
    Timestamps that are further away in time from the `from_time` variable than that designated
    by `fail_span` are flagged as FAIL. Data points that are newer than `from_time` are not considered
    for this test and are passed through as GOOD. Any missing and masked data is flagged as UNKNOWN.

    Parameters
    ----------
    tinp
        Time input data as a numeric numpy array or list of real numbers.
    fail_span
        A numeric value in hours which, if data is older than, is flagged as FAIL.
    from_time
        A timestamp which, if defined, is used to anchor measurements against. Defaults to current date and time. [optional]

    Returns
    -------
    flag_arr
        A masked array of flag values equal in size to that of the input `tinp`.
    """
    #   Init 
    if from_time == None:
        from_time = np.datetime64("now")
    else:
        from_time = np.datetime64(from_time)
    
    original_shape = tinp.shape
    tinp = np.ma.asarray(tinp, dtype="datetime64[ns]").flatten()
    flag_arr = np.ma.ones(tinp.size, dtype="uint8") #   Init to flag 1 "good"
    
    tinp.mask = np.isnat(tinp.data)
    flag_arr[tinp.mask] = QartodFlags.MISSING   #   Init missing timestamps to the missing flag
    valid = ~tinp.mask  #   Define where the data point are not missing, such that we can index them and keep those flags.
    
    fail_span = np.timedelta64(int(fail_span * 3600), "s")
    diff_time = from_time - tinp.data
    flag_arr[valid & (diff_time > fail_span)] = QartodFlags.FAIL
    
    return flag_arr.reshape(original_shape) 

Let's test it with an arbitrary value inside the dataset, using the default 6 hour tolerance for flagging. Everything after this 6 hour tolerance should be flagged as 1, whereas everything before it should be flagged as 4.

In [61]:
from_time = np.datetime64('2026-02-10T11:17:07.000000000')
print(np.where(ds.TIME == from_time))

(array([1690309]),)


In [60]:
flags = data_reception_test(ds.TIME.to_numpy(), from_time=from_time)
print(f"The last index where data are flagged: {np.where(flags == 4)[0][-1]}")
print(f"Length of the array: {len(ds.TIME)}")

The last index where data are flagged: 1669201
Length of the array: 1690311


In [ ]:
print(flags[1669195:1669210])   #   Looks to be index 1669201 is just over 6 hours away.
print(np.datetime64(from_time - ds.TIME[1669202].item()))   #   Returns just 6 hours from Jan 1, 1970

[4 4 4 4 4 4 4 1 1 1 1 1 1 1 1]
1970-01-01T06:00:00.000000000


#### time_gap_test()
This is the first test that probably *shouldn't* be used on NRT data, as it looks at the gaps within the data itself. Similar to the previous test, given a threshold, it takes the diff of the time array and flags the point *after* any gaps that are exceeding the specified `fail_span`.

In [ ]:
def time_gap_test(
    tinp: Sequence[Real],
    fail_span: Real,
) -> np.ma.core.MaskedArray:
    """

    Parameters
    ----------
    tinp
        Input data (of timestamps) as a list-like sequence or array-like format.
    fail_span
        The FAIL threshold value, expressed in hours.
    
    Returns
    -------
    flag_arr
        A Numpy masked array of flag values equal in size to that of the input
    """
    
    flags = np.full((1, len(tinp)-1), NOTEVAL_VALUE)
    
    pass


In [ ]:
QartodFlags